<h2><b><center><span style="color:blue">LANGGRAPH

- LangGraph is an <span style="color: limegreen;">orchestration framework</span> for applications that need multiple steps, shared state, branching, loops, and recovery. It lets you combine predictable Python logic with LLM-driven decisions.

### <span style="color: magenta;">1. Where LangGraph Fits?</span>

| <span style="color: magenta;"><center>Component</span> | <span style="color: magenta;"><center>Main responsibility</span> |
|-|-|
| <span style="color: limegreen;">LLM</span> | Interpret input, generate output, or propose actions|
| <span style="color: limegreen;">LangChain</span> | Model/tool integrations and higher-level agent building|
| <span style="color: limegreen;">LangGraph</span> | Control execution, state, routing, and recovery |
| <span style="color: limegreen;">Deep Agents</span> | Add planning, subagents, files, and context management |
| <span style="color: limegreen;">LangSmith</span> | Trace, evaluate, debug, and deploy applications|

<br>

- We can call existing model API client inside LangGraph nodes. LangChain model wrappers and LangSmith are optional for using the core framework.

### <span style="color: magenta;">2. The theory: a graph-based state machine</span>

- A graph contains <span style="color: limegreen;">nodes</span> connected by <span style="color: limegreen;">directed edges</span>. The direction indicates where execution can proceed.
- A <span style="color: limegreen;">state machine</span> changes its working data and chooses its next action according to rules.
- LangGraph combines these ideas: <span style="color: limegreen;">nodes process data</span>, <span style="color: limegreen;">update state</span>, and <span style="color: limegreen;">trigger subsequent nodes</span>.
- Unlike a DAG, which cannot contain cycles, LangGraph <span style="color: limegreen;">supports cycles</span>. This enables correction loops and repeated tool use.
- The graph defines permitted behavior. Whether a transition is chosen by Python or an LLM depends on your design.

### <span style="color: magenta;">3. State, nodes, and edges</span>

| <span style="color: magenta;"><center>Concept</span>          | <span style="color: magenta;"><center>Meaning</span>                                | <span style="color: magenta;"><center>Email extraction example</span>              |
| ---------------- | -------------------------------------- | ------------------------------------- |
| State            | Working data carried through execution | Email, amounts, periods, errors       |
| Node             | Function performing a step             | Extract amounts                       |
| Edge             | Connection between steps               | Amount extraction → period extraction |
| Conditional edge | Route selected from current state      | Valid → save; ambiguous → review      |
| `START` / `END`  | Entry and completion markers           | Begin processing / finish             |

<br>

- A node can use regex, SQL, an API, or an LLM. A node is not automatically an agent.

### <span style="color: magenta;">4. State schemas and data contracts</span>

- A <span style="color: limegreen;">schema</span> describes fields and expected types. Common choices are `TypedDict`, dataclasses, and Pydantic.
- `TypedDict` supplies <span style="color: limegreen;">type hints</span>; it <span style="color: limegreen;">does not validate values at runtime</span>. <span style="color: limegreen;">Pydantic supports runtime validation</span>.
- You can <span style="color: limegreen;">separate input, internal, and output schemas</span>. This keeps intermediate data out of the returned result.
- The <span style="color: limegreen;">graph-state schema</span> defines <span style="color: limegreen;">workflow data</span>. An <span style="color: limegreen;">LLM-output schema</span> defines the <span style="color: limegreen;">response expected from the model</span>.
- <span style="color: limegreen;">Correct structure does not prove factual correctness</span>: amount=5000 can satisfy the schema while contradicting the email.
- Keep checkpointed data serializable. Pass clients and connections through appropriate runtime dependencies.

### <span style="color: magenta;">5. Nodes and partial updates</span>

- A node <span style="color: limegreen;">reads state</span> and <span style="color: limegreen;">returns the fields it changes</span>. Other fields remain available.
- Return updates rather than directly mutating the input state.
- Example: an amount node returns {"amounts": [...]}; it need not return the email again.
- Separate steps when they have different retry, review, or debugging requirements.
- <span style="color: limegreen;">Excessively large nodes</span> repeat <span style="color: limegreen;">more work after failure</span>. <span style="color: limegreen;">Excessively fragmented nodes</span> add <span style="color: limegreen;">complexity</span>. Choose boundaries around meaningful operations.

### <span style="color: magenta;">6. Reducers: combining updates</span>

- By default, a new value replaces the existing field value.
- A reducer <span style="color: limegreen;">defines how old and new values combine</span>. List concatenation is one example.
- With operator.add, ["error A"] plus ["error B"] becomes ["error A", "error B"].
- It does not deduplicate. Return only new items when using an append reducer.
- Returning [] to an append field does not clear it. Overwrite can bypass the reducer.
- Use replacement for revised extraction results; accumulation for genuinely additive information.

### <span style="color: magenta;">7. Messages and conversation history</span>

- Agent applications often store human, AI, and tool messages in state.
- `MessagesState` provides a ready-made messages field.
- Its `add_messages` reducer <span style="color: limegreen;">appends new messages and updates existing ones by message ID</span>.
- Plain list concatenation cannot handle message replacement correctly.
- Stored state is not automatically sent to the LLM. <span style="color: limegreen;">Your model-calling node decides which history and fields enter the request</span>.